# Catalogue exploration

This notebook only calls the package under `src/`. Cleaning, the soup, and the model live there and in `scripts/`. Re-run `scripts/profile_data.py` for the full profile and `scripts/evaluate.py` for metrics.

The file on disk has 10 columns. `cast` and `description` are absent. Missing director and country values are the text `Not Given`, not pandas nulls.

In [1]:
from src.config import load_config
from src.data.clean import clean_frame
from src.data.load import load_raw_csv
from src.data.repairs import MISSING_SENTINELS
from src.features.text import build_soup

config = load_config()
raw = load_raw_csv(config.raw_csv)
print(f"{raw.shape[0]} rows, {raw.shape[1]} columns")
print(list(raw.columns))

Optional column 'cast' is absent from netflix_titles.csv; text features will leave it empty


Optional column 'description' is absent from netflix_titles.csv; text features will leave it empty


8790 rows, 12 columns
['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'cast', 'description']


In [2]:
for column in ("director", "country", "rating"):
    text = raw[column].fillna("").astype(str).str.strip().str.lower()
    sentinel = int(text.isin(MISSING_SENTINELS).sum())
    print(f"{column}: {sentinel} sentinel rows ({sentinel / len(raw):.1%})")

director: 2588 sentinel rows (29.4%)
country: 287 sentinel rows (3.3%)
rating: 0 sentinel rows (0.0%)


In [3]:
cleaned = clean_frame(raw)
print(cleaned.groupby("type")[["duration_minutes", "duration_seasons"]].agg(["count", "median"]).to_string())
print("rating values that look like a duration after cleaning:", int(cleaned["rating"].fillna("").str.match(r"^\d+\s*min$").sum()))

        duration_minutes        duration_seasons       
                   count median            count median
type                                                   
Movie               6126   98.0                0    NaN
TV Show                0    NaN             2664    1.0
rating values that look like a duration after cleaning: 0


In [4]:
genres = cleaned["genres"].explode()
print(genres.value_counts().head(15).to_string())

genres
International Movies        2752
Dramas                      2426
Comedies                    1674
International TV Shows      1349
Documentaries                869
Action & Adventure           859
TV Dramas                    762
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Thrillers                    577
TV Comedies                  573
Crime TV Shows               469
Kids' TV                     448
Docuseries                   394


In [5]:
weights = dict(config.feature_weights)
for title in ("Breaking Bad", "The Irishman", "ROMA"):
    row = cleaned.loc[cleaned["title"] == title].iloc[0]
    soup = build_soup(row, weights, config.recommender.cast_cap)
    print(title)
    print(soup)
    print("---")

Breaking Bad
crimetvshows tvdramas tvthrillers crimetvshows tvdramas tvthrillers crimetvshows tvdramas tvthrillers tvshow tvma crime tv shows tv dramas tv thrillers
---
The Irishman
dramas dramas dramas martinscorsese martinscorsese movie r dramas
---
ROMA
dramas independentmovies internationalmovies dramas independentmovies internationalmovies dramas independentmovies internationalmovies alfonsocuaron alfonsocuaron movie r dramas independent movies international movies
---


The soup repeats the collapsed genre token three times and the spaced genre words once. Country weight is 0, so a country name does not appear. The Irishman and ROMA keep the collapsed director token (`martinscorsese`, `alfonsocuaron`). Breaking Bad has no director in this extract, so that token is absent rather than the word `unknown`.

There is no watch history, so the notebook does not report an accuracy.